# V20 – Praxis: Musterlösungen Fill-in-Blanks

Dies ist die Lösung der drei Fill-in-Blanks aus [02_praxis.ipynb](../02_praxis.ipynb).

## 📦 Voraussetzungen
- `../02_praxis.ipynb` selbst durchgearbeitet.


## `setup_db()` – identisch zum Praxis-Notebook

In [1]:
import sqlite3

def setup_db() -> sqlite3.Connection:
    """Legt eine frische In-Memory-DB mit deterministischen Daten an."""
    conn = sqlite3.connect(":memory:")
    cur = conn.cursor()
    cur.execute("""
        CREATE TABLE Maschinen (
            Maschinen_ID INTEGER PRIMARY KEY,
            Name         TEXT NOT NULL,
            Typ          TEXT NOT NULL,
            Baujahr      INTEGER NOT NULL
        )
    """)
    cur.execute("""
        CREATE TABLE Wartungen (
            Wartung_ID    INTEGER PRIMARY KEY,
            Maschinen_ID  INTEGER NOT NULL,
            Wartungstyp   TEXT NOT NULL,
            Datum         TEXT NOT NULL,
            Kosten        REAL NOT NULL,
            FOREIGN KEY (Maschinen_ID) REFERENCES Maschinen(Maschinen_ID)
        )
    """)
    maschinen = [
        (1, "CNC-Fräse 01",    "Fräse",    2018),
        (2, "CNC-Fräse 02",    "Fräse",    2020),
        (3, "Drehmaschine 01", "Drehbank", 2016),
        (4, "Drehmaschine 02", "Drehbank", 2021),
        (5, "Roboter-Arm R1",  "Roboter",  2019),
    ]
    wartungen = [
        (1, 1, "Inspektion",   "2024-01-10",  150.00),
        (2, 1, "Reparatur",    "2024-03-05", 1200.00),
        (3, 2, "Inspektion",   "2024-02-14",  180.00),
        (4, 3, "Ölwechsel",    "2024-01-20",   90.00),
        (5, 3, "Reparatur",    "2024-04-12", 2500.00),
        (6, 3, "Inspektion",   "2024-05-18",  200.00),
        (7, 4, "Inspektion",   "2024-02-28",  160.00),
        (8, 5, "Kalibrierung", "2024-03-22",  450.00),
    ]
    cur.executemany("INSERT INTO Maschinen VALUES (?, ?, ?, ?)", maschinen)
    cur.executemany("INSERT INTO Wartungen VALUES (?, ?, ?, ?, ?)", wartungen)
    conn.commit()
    return conn

# Kurztest
conn = setup_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM Maschinen")
print("Maschinen:", cur.fetchone()[0])
cur.execute("SELECT COUNT(*) FROM Wartungen")
print("Wartungen:", cur.fetchone()[0])
conn.close()


Maschinen: 5
Wartungen: 8


## Lösung Fill-in 1 – `COUNT`

In [2]:
conn = setup_db()
cur = conn.cursor()
cur.execute("SELECT COUNT(*) FROM Wartungen")
anzahl = cur.fetchone()[0]
conn.close()

assert anzahl == 8
print("Ergebnis:", anzahl)


Ergebnis: 8


## Lösung Fill-in 2 – `SUM` + `GROUP BY`

SQLite sortiert `ORDER BY Wartungstyp` lexikografisch; „Ölwechsel" landet hinter den ASCII-Buchstaben.

In [3]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT Wartungstyp, SUM(Kosten) AS gesamt
    FROM Wartungen
    GROUP BY Wartungstyp
    ORDER BY Wartungstyp
""")
ergebnis = cur.fetchall()
conn.close()

assert ergebnis == [
    ("Inspektion",    690.0),
    ("Kalibrierung",  450.0),
    ("Reparatur",    3700.0),
    ("Ölwechsel",      90.0),
]
for zeile in ergebnis:
    print(zeile)


('Inspektion', 690.0)
('Kalibrierung', 450.0)
('Reparatur', 3700.0)
('Ölwechsel', 90.0)


## Lösung Fill-in 3 – `INNER JOIN` + `MAX`

In [4]:
conn = setup_db()
cur = conn.cursor()
cur.execute("""
    SELECT M.Name, W.Kosten
    FROM Maschinen M
    INNER JOIN Wartungen W ON M.Maschinen_ID = W.Maschinen_ID
    ORDER BY W.Kosten DESC
    LIMIT 1
""")
teuerste = cur.fetchone()
conn.close()

assert teuerste == ("Drehmaschine 01", 2500.0)
print("Teuerste einzelne Wartung:", teuerste)


Teuerste einzelne Wartung: ('Drehmaschine 01', 2500.0)


## ✅ Fertig

Zurück zu [../03_aufgaben.ipynb](../03_aufgaben.ipynb).
